In [12]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/GOA_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)

features = ['NDVI', 'NDMI', 'NBR', 'BSI']
df = df[['latitude', 'longitude', 'year'] + features]
df = df.dropna()
df = df.sort_values(by=['latitude', 'longitude', 'year'])

# ===============================
# 3. Create Sequences (RAW DATA)
# ===============================
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

for (lat, lon), group in df.groupby(['latitude', 'longitude']):
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/GOA_Deforestation_Predictions.csv',
    index=False
)

print("Saved: GOA_Deforestation_Predictions.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw input shape: (28507, 4, 4)

Label distribution:
No deforestation: 26051
Deforestation: 2456
Train shape: (22805, 4, 4)
Test shape : (5702, 4, 4)
Scaled train shape: (22805, 4, 4)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,777 (77.25 KB)

 Trainable params: 19,777 (77.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.1817 - loss: 1.3431 - precision: 0.0860 - recall: 0.8847 - val_accuracy: 0.0933 - val_loss: 0.7883 - val_precision: 0.0866 - val_recall: 0.9980
Epoch 2/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.4103 - loss: 1.2991 - precision: 0.1104 - recall: 0.8047 - val_accuracy: 0.5714 - val_loss: 0.6807 - val_precision: 0.1565 - val_recall: 0.9063
Epoch 3/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6682 - loss: 1.0328 - precision: 0.1807 - recall: 0.8218 - val_accuracy: 0.6687 - val_loss: 0.5887 - val_precision: 0.2005 - val_recall: 0.9532
Epoch 4/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7679 - loss: 0.8090 - precision: 0.2543 - recall: 0.8770 - val_accuracy: 0.8024 - val_loss: 0.4103 - val_precision: 0.2948 - val_recall: 0.9308
Epoch 5/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8224 - loss: 0.6516 - precision: 0.3096 - recall: 0.9059 - val_accuracy: 0.7361 - val_loss: 

In [13]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/GOA_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/GOA_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/GOA_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


NDVI: {'mean': np.float64(-0.2673416006728172), 'std': 0.03524883219843842, 'q1': np.float64(-0.28999094000000003), 'q3': np.float64(-0.250374565), 'lower_2std': np.float64(-0.33783926506969403), 'upper_2std': np.float64(-0.19684393627594038)}
NBR : {'mean': np.float64(-0.19744072705279408), 'std': 0.033509890142069655, 'q1': np.float64(-0.21940841999999997), 'q3': np.float64(-0.177791255), 'lower_2std': np.float64(-0.2644605073369334), 'upper_2std': np.float64(-0.1304209467686548)}
BSI : {'mean': np.float64(0.10586964667028266), 'std': 0.02824399727222159, 'q1': np.float64(0.0896430949697492), 'q3': np.float64(0.12359816503729529), 'lower_2std': np.float64(0.04938165212583948), 'upper_2std': np.float64(0.16235764121472585)}
NDMI: {'mean': np.float64(-0.07973332252070334), 'std': 0.027904693373755427, 'q1': np.float64(-0.097148), 'q3': np.float64(-0.06295676499999998), 'lower_2std': np.float64(-0.13554270926821418), 'upper_2std': np.float64(-0.023923935773192483)}

Deforestation Cause 

In [15]:
# Goa roughly: latitude 14.9–15.8, longitude 73.7–74.3
m = folium.Map(location=[15.4, 74.0], zoom_start=10)
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Load improved cause CSV
df = pd.read_csv('/content/drive/MyDrive/GOA_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())


cause
Logging        842
Agriculture    310
Fire            61
Urban/Other     23
Name: count, dtype: int64


In [16]:
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}
# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)


In [17]:
m

In [18]:
m.save('/content/drive/MyDrive/GOA_Deforestation_Causes_Adaptive_Map.html')


In [19]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
df_def = pd.read_csv(
    '/content/drive/MyDrive/GOA_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))
df_cause = pd.read_csv(
    '/content/drive/MyDrive/GOA_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())


Total samples: 5702
    latitude  longitude  deforestation
0  15.331098  74.313581              0
1  14.992433  74.220156              1
2  15.023874  74.236326              1
3  15.289775  74.279445              0
4  15.595202  74.252496              0
Total deforestation samples: 1236
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  14.992433  74.220156              1    -0.292946   -0.229606    0.124504   
1  15.023874  74.236326              1    -0.311179   -0.231314    0.123668   
2  15.649101  74.226444              1    -0.315292   -0.254813    0.164803   
3  15.648203  74.203987              1    -0.309762   -0.241203    0.142687   
4  15.060705  74.152783              1    -0.293252   -0.221854    0.126120   

   NDMI_change    cause  
0    -0.095550  Logging  
1    -0.104989  Logging  
2    -0.136186  Logging  
3    -0.115539  Logging  
4    -0.102202  Logging  


In [20]:
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# MERGE CAUSE DATA
# ==============================
df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())

# ==============================
# CREATE POINT GEOMETRY
# ==============================
geometry = [
    Point(xy) for xy in zip(
        df_deforest['longitude'],
        df_deforest['latitude']
    )
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

# ==============================
# LOAD GOA DISTRICTS
# ==============================
districts = gpd.read_file(
    '/content/drive/MyDrive/Goa.topojson'
)

# ⚠ CRS safety check
if districts.crs is None:
    districts = districts.set_crs("EPSG:4326")

districts = districts.to_crs("EPSG:4326")

print(districts.head())


    latitude  longitude  deforestation    cause
0  14.992433  74.220156              1  Logging
1  15.023874  74.236326              1  Logging
2  15.649101  74.226444              1  Logging
3  15.648203  74.203987              1  Logging
4  15.060705  74.152783              1  Logging
     id REMARKS_2 Country State_Name State_Code  Dist_Name Dist_Code  \
0  None      None   India        Goa         30  North Goa       585   
1  None      None   India        Goa         30  South Goa       586   

                                            geometry  
0  POLYGON ((74.25633 15.48951, 74.23732 15.4832,...  
1  MULTIPOLYGON (((73.78178 15.35566, 73.78178 15...  


In [22]:
# ==============================
# SPATIAL JOIN
# ==============================
gdf_joined = gpd.sjoin(
    gdf_points,
    districts,
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())

print(gdf_joined['Dist_Name'].value_counts())

# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================
if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# Optional cleanup
gdf_joined.drop(columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
                inplace=True)

# ==============================
# DISTRICT SUMMARY
# ==============================
district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/GOA_District_Wise_Deforestation.csv',
    index=False
)

print("Saved district summary")

# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================
cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/GOA_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved cause summary")


    latitude  longitude  deforestation    cause                   geometry  \
0  14.992433  74.220156              1  Logging  POINT (74.22016 14.99243)   
1  15.023874  74.236326              1  Logging  POINT (74.23633 15.02387)   
2  15.649101  74.226444              1  Logging   POINT (74.22644 15.6491)   
3  15.648203  74.203987              1  Logging   POINT (74.20399 15.6482)   
4  15.060705  74.152783              1  Logging   POINT (74.15278 15.0607)   

   index_right    id REMARKS_2 Country State_Name State_Code  Dist_Name  \
0            1  None      None   India        Goa         30  South Goa   
1            1  None      None   India        Goa         30  South Goa   
2            0  None      None   India        Goa         30  North Goa   
3            0  None      None   India        Goa         30  North Goa   
4            1  None      None   India        Goa         30  South Goa   

  Dist_Code  
0       586  
1       586  
2       585  
3       585  
4       58

In [24]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Goa District Boundary
# =====================================================
goa = gpd.read_file('/content/drive/MyDrive/Goa.topojson')

# CRS safety
if goa.crs is None:
    goa.set_crs(epsg=4326, inplace=True)

goa = goa.to_crs(epsg=4326)
goa["state"] = "Goa"
gdf_districts = goa.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/GOA_Deforestation_Predictions.csv")

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# =====================================================
# STEP 7: Merge Stats
# =====================================================
gdf_districts = gdf_districts.merge(
    district_total, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_deforestation, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_afforestation, on="Dist_Name", how="left"
)

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 8: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

# Deforestation
gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

# Afforestation
gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 9: Create Folium Map (center Goa)
# =====================================================
m = folium.Map(location=[15.4, 74.0], zoom_start=10)

# =====================================================
# STEP 10: Dynamic Choropleth
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Goa)"
).add_to(m)

# =====================================================
# STEP 11: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 12: Dynamic Goa Afforestation Alert Popup
# =====================================================
state_def = gdf_districts["deforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Goa Afforestation Alert</b><br><br>

Total Deforestation Points: <b>{int(state_def)}</b><br><br>

High Impact Districts:<br>
{top_districts_html}<br>

🌱 Immediate afforestation drives recommended.<br>
Focus on plantation, eco-restoration,<br>
and sustainable land practices.
"""

folium.Marker(
    location=[15.4, 74.0],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 13: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/goa_forest.html")

print("✅ Goa map saved successfully with afforestation recommendation!")

/tmp/ipython-input-346/291000616.py:88: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Goa map saved successfully with afforestation recommendation!
